In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import requests

# 1. Define tu token personal (reemplaza 'TU_TOKEN' con el código de tu cuenta del Banco Central)
import os
from dotenv import load_dotenv
load_dotenv()
mi_token = os.getenv('BCCH_TOKEN')

# 2. Crea una función para no repetir código por cada dato que necesites
def obtener_dato_bcentral(codigo_serie, token):
    # Armar el enlace de la API
    url = f"https://si3.bcentral.cl/SieteRestWS/SieteRestWS.ashx?token={token}&timeseries={codigo_serie}&function=GetSeries"
    
    # Hacer la petición web
    respuesta = requests.get(url)
    datos = respuesta.json()
    
    # Extraer las observaciones del JSON a un DataFrame
    df = pd.DataFrame(datos['Series']['Obs'])
    
    # Formatear las columnas: convertir fecha a datetime y el valor a número
    df['fecha'] = pd.to_datetime(df['indexDateString'], format='%d-%m-%Y')
    df['valor'] = pd.to_numeric(df['value'], errors='coerce')
    
    # Filtrar solo las columnas útiles y renombrar el valor con el nombre de la serie
    df = df[['fecha', 'valor']].rename(columns={'valor': codigo_serie})
    
    return df

# 3. Descargar las dos series que necesitas usando sus códigos API
# (Uso los códigos de tu imagen como ejemplo)
df_ipc_general = obtener_dato_bcentral('F074.IPC.IND.Z.EP23.C.M', mi_token)
df_ipc_alimentos = obtener_dato_bcentral('F074.IPCA.IND.Z.EP23.Z.M', mi_token)

# 4. Unir ambas bases de datos respetando todas las fechas (outer join)
df_maestra = pd.merge(df_ipc_general, df_ipc_alimentos, on='fecha', how='outer')

# 5. Ordenar por fecha para mayor claridad
df_maestra = df_maestra.sort_values('fecha').reset_index(drop=True)

print(df_maestra)

         fecha  F074.IPC.IND.Z.EP23.C.M  F074.IPCA.IND.Z.EP23.Z.M
0   2009-12-01                    57.31                     42.65
1   2010-01-01                    57.61                     42.39
2   2010-02-01                    57.77                     42.31
3   2010-03-01                    57.82                     42.69
4   2010-04-01                    58.09                     42.60
..         ...                      ...                       ...
197 2026-05-01                   112.37                    110.74
198 2026-06-01                   112.35                    111.60
199 2026-07-01                   112.45                    112.37
200 2026-08-01                   113.15                    113.98
201 2026-09-01                   113.59                    114.66

[202 rows x 3 columns]


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
mi_token = os.getenv('BCCH_TOKEN')

In [4]:
# Guarda los datos en una base SQLite con 3 tablas: temas, series y observaciones
engine = create_engine('sqlite:///data/portafolio.db')

temas = pd.DataFrame({'id_tema': [2], 'tema': ['Precios']})
series = pd.DataFrame({
    'id_serie': ['F074.IPC.IND.Z.EP23.C.M', 'F074.IPCA.IND.Z.EP23.Z.M'],
    'nombre':   ['IPC general', 'IPC alimentos'],
    'id_tema':  [2, 2],
})
# Pasa df_maestra a formato largo: una fila por serie y fecha
observaciones = df_maestra.melt(id_vars='fecha', var_name='id_serie', value_name='valor').dropna()
observaciones['fecha'] = observaciones['fecha'].dt.strftime('%Y-%m-%d')

temas.to_sql('temas', engine, if_exists='replace', index=False)
series.to_sql('series', engine, if_exists='replace', index=False)
observaciones.to_sql('observaciones', engine, if_exists='replace', index=False)
print('Observaciones guardadas:', len(observaciones))

Observaciones guardadas: 404


In [5]:
# Muestra la tabla observaciones
pd.read_sql("SELECT * FROM observaciones", engine)

,fecha,id_serie,valor
0,2009-12-01,F074.IPC.IND.Z.EP23.C.M,57.31
1,2010-01-01,F074.IPC.IND.Z.EP23.C.M,57.61
2,2010-02-01,F074.IPC.IND.Z.EP23.C.M,57.77
3,2010-03-01,F074.IPC.IND.Z.EP23.C.M,57.82
4,2010-04-01,F074.IPC.IND.Z.EP23.C.M,58.09
...,...,...,...
399,2026-05-01,F074.IPCA.IND.Z.EP23.Z.M,110.74
400,2026-06-01,F074.IPCA.IND.Z.EP23.Z.M,111.60
401,2026-07-01,F074.IPCA.IND.Z.EP23.Z.M,112.37
402,2026-08-01,F074.IPCA.IND.Z.EP23.Z.M,113.98


In [6]:
#muestra la tabla series
pd.read_sql("SELECT * FROM series", engine)

,id_serie,nombre,id_tema
0,F074.IPC.IND.Z.EP23.C.M,IPC general,2
1,F074.IPCA.IND.Z.EP23.Z.M,IPC alimentos,2


In [7]:
#muestra la tabla temas
pd.read_sql("SELECT * FROM temas", engine)

,id_tema,tema
0,2,Precios


In [8]:
# JOIN: une las 3 tablas para ver cada dato con su nombre y tema
consulta_join = """
SELECT t.tema, s.nombre, o.fecha, o.valor
FROM observaciones AS o
JOIN series AS s ON s.id_serie = o.id_serie
JOIN temas  AS t ON t.id_tema  = s.id_tema
ORDER BY o.fecha DESC
LIMIT 10
"""
pd.read_sql(consulta_join, engine)

,tema,nombre,fecha,valor
0,Precios,IPC general,2026-09-01,113.59
1,Precios,IPC alimentos,2026-09-01,114.66
2,Precios,IPC general,2026-08-01,113.15
3,Precios,IPC alimentos,2026-08-01,113.98
4,Precios,IPC general,2026-07-01,112.45
5,Precios,IPC alimentos,2026-07-01,112.37
6,Precios,IPC general,2026-06-01,112.35
7,Precios,IPC alimentos,2026-06-01,111.60
8,Precios,IPC general,2026-05-01,112.37
9,Precios,IPC alimentos,2026-05-01,110.74


In [9]:
# GROUP BY: promedio anual de cada serie
consulta_group = """
SELECT s.nombre,
       strftime('%Y', o.fecha) AS anio,
       ROUND(AVG(o.valor), 2)  AS promedio_anual
FROM observaciones AS o
JOIN series AS s ON s.id_serie = o.id_serie
GROUP BY s.nombre, anio
ORDER BY s.nombre, anio
"""
pd.read_sql(consulta_group, engine)

,nombre,anio,promedio_anual
0,IPC alimentos,2009,42.65
1,IPC alimentos,2010,43.58
2,IPC alimentos,2011,46.52
3,IPC alimentos,2012,50.09
4,IPC alimentos,2013,52.38
5,IPC alimentos,2014,56.07
6,IPC alimentos,2015,60.09
7,IPC alimentos,2016,62.29
8,IPC alimentos,2017,63.50
9,IPC alimentos,2018,65.48


In [10]:
# Función de ventana: calcula la inflación a 12 meses comparando cada mes con el mismo mes del año anterior
consulta_ventana = """
SELECT s.nombre,
       o.fecha,
       o.valor,
       ROUND(100.0 * (o.valor / LAG(o.valor, 12) OVER (
                 PARTITION BY o.id_serie ORDER BY o.fecha) - 1), 2) AS inflacion_12m
FROM observaciones AS o
JOIN series AS s ON s.id_serie = o.id_serie
ORDER BY s.nombre, o.fecha
"""
pd.read_sql(consulta_ventana, engine).tail(12)

,nombre,fecha,valor,inflacion_12m
392,IPC general,2025-10-01,109.19,3.44
393,IPC general,2025-11-01,109.47,3.44
394,IPC general,2025-12-01,109.26,3.45
395,IPC general,2026-01-01,109.71,2.78
396,IPC general,2026-02-01,109.70,2.37
397,IPC general,2026-03-01,110.75,2.83
398,IPC general,2026-04-01,112.18,3.96
399,IPC general,2026-05-01,112.37,3.93
400,IPC general,2026-06-01,112.35,4.34
401,IPC general,2026-07-01,112.45,3.53


In [11]:
# Revisión de la base: cuántos datos tiene cada serie y entre qué fechas, para detectar errores de carga
consulta_revision = """
SELECT t.tema,
       s.nombre,
       COUNT(o.valor) AS n_obs,
       MIN(o.fecha)   AS desde,
       MAX(o.fecha)   AS hasta
FROM temas AS t
LEFT JOIN series AS s        ON s.id_tema  = t.id_tema
LEFT JOIN observaciones AS o ON o.id_serie = s.id_serie
GROUP BY t.tema, s.nombre
ORDER BY t.tema, n_obs
"""
pd.read_sql(consulta_revision, engine)

,tema,nombre,n_obs,desde,hasta
0,Precios,IPC alimentos,202,2009-12-01,2026-09-01
1,Precios,IPC general,202,2009-12-01,2026-09-01


In [12]:
# Control de calidad: cuenta filas repetidas (misma serie y fecha); debe dar 0
consulta_duplicados = """
SELECT COUNT(*) AS duplicados
FROM (SELECT id_serie, fecha
      FROM observaciones
      GROUP BY id_serie, fecha
      HAVING COUNT(*) > 1)
"""
pd.read_sql(consulta_duplicados, engine)

,duplicados
0,0
